# పాఠం 08 - బహుళ ఏజెంట్ డిజైన్ ప్యాటర్న్


## సెటప్


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## మల్టీ-ఏజెంట్ సిస్టమ్స్ ఎందుకు?

ప్రయాణ ఏర్పాట్ల వంటి వాస్తవ ప్రపంచ పనులు అనేక రకాల నైపుణ్యాలను అవసరం చేస్తాయి — లాజిస్టిక్స్, స్థానిక జ్ఞానం, బడ్జెటింగ్, మరియు మరిన్ని. ఒకే ఏజెంట్ అన్నిటినీ నిర్వహించడానికి ప్రయత్నిస్తే అది త్వరగా అసహ్యం అవుతుంది.

మల్టీ-ఏజెంట్ సిస్టమ్స్ దీనిని **ప్రత్యేకత** ద్వారా పరిష్కరిస్తాయి: ప్రతి ఏజెంట్ ఒక ప్రత్యేక నైపుణ్యంపై దృష్టి పెడతాడు, సాధారణ ఏజెంట్ కంటే మెరుగైన నాణ్యత ఫలితాలను అందిస్తాడు. అవి **స్థాయిలో పెరుగుదల** కూడా మెరుగుపరుస్తాయి — మీరు కొత్త ఏజెంట్లను (ఉదా: ఎయిర్ ఫ్లైట్ నిపుణుడు, రెస్టారెంట్ సమీక్షకుడు) ప్రస్తుత వర్క్‌ఫ్లో మార్చకుండా జోడించవచ్చు. ఏజెంట్లు ఒక నిర్మిత పైప్‌లైన్ ద్వారా కలసికట్టుగా పనిచేస్తారు, ఒకరిద్దరికి సందర్భం అందిస్తూ.


## ప్రత్యేక ఏజెంట్ల సృష్టి


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ఒక క్రమబద్ధమైన వర్క్‌ఫ్లోని నిర్మించడం

`WorkflowBuilder` మీరు ఏజెంట్లనుirected గ్రాఫ్‌లో కనెక్ట్ చేయడానికి అనుమతిస్తుంది. ఇక్కడ మనము ఒక సాదా రెండు దశల పైప్‌లైన్‌ను సృష్టిస్తాము: **TravelPlanner** ప్రయాణ ప్రయోజన ప్రణాళికను రూపొందిస్తుంది, తర్వాత **TravelConcierge** దాన్ని సమీక్షించి మెరుగుపరుస్తుంది.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## వర్క్‌ఫ్లోలో మరిన్ని ఏజెంట్లను జోడించడం

మల్టీ ఏజెంట్ ప్యాటర్న్ యొక్క ముఖ్యమైన లాభాలలో ఒకటి, దాన్ని ఎంత సులభంగా విస్తరించగలవో. క్రింద, మేము ఒక **BudgetReviewer** ఏజెంట్‌ని జోడించాము, ఇది ప్రయాణికుడి బడ్జెట్‌ను దృష్టిలో ఉంచుకుని ప్రణాళికను తనిఖీ చేస్తుంది, ఖర్చులను పరిమితికి మించి పోవొచ్చని ఆశంకాజనకమైన అంశాలను గుర్తిస్తుంది, మరియు డబ్బు ఆదా చేసే ప్రత్యామ్నాయాలను సూచిస్తుంది. ఈ వర్క్‌ఫ్లో ఇప్పుడు మూడు ఏజెంట్లను వరుసగా నడుపుతుంది:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## సారాంశం

ఈ పాఠంలో మీరు నేర్చుకున్నది:

1. **ప్రత్యేక ఏజెంట్లను సృష్టించడం** — ప్రతి ఒక్కటి ఒక ప్రత్యేక పాత్ర (ప్లానింగ్, కన్సియర్జ్, బడ్జెట్ సమీక్ష) కోసం.
2. **ఏజెంట్లను పరస్పర అనుబంధ వర్క్‌ఫ్లోలో కనెక్ట్ చేయడం** `WorkflowBuilder` మరియు `add_edge` ఉపయోగించి.
3. **బహుళ ఏజెంట్ల పైప్‌లైన్ నుండి అవుట్‌పుట్‌ను స్ట్రీమ్ చేయడం**, ఏ ఏజెంట్ మాట్లాడుతున్నదో ట్రాక్ చేస్తూ.
4. **ఓ వర్క్‌ఫ్లోను విస్తరించడం** అర్థం మార్చకుండా కొత్త ఏజెంట్లను చేర్చడం.

బహుళ ఏజెంట్ డిజైన్ నమూనా ప్రతి ఏజెంట్‌ను సాదారణంగా ఉంచి, ఏ ఒక్క ఏజెంట్ కూడా సాధించలేని సమగ్ర, బాగా సమీక్షించిన ఫలితాలను ఉత్పత్తి చేస్తుంది.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**హితసం**:  
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మనం ఖచ్చితత్వానికి ప్రయత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాలలో పొరపాట్లు లేదా తప్పిదాలు ఉండవచ్చు. మూల పత్రం దాని స్వదేశీ భాషలో అధికారిక మూలంగా పరిగణించాలి. ముఖ్యమైన సమాచారానికి, వృత్తిపరమైన మానవ అనువాదం సిఫారసు చేయబడుతుంది. ఈ అనువాదం వల్ల జరిగే ఏవైనా అపార్థాలు లేదా తప్పు అర్థాలు వచ్చినట్లయితే మేము బాధ్యత వహించడం లేదు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
